# GNN 266 — Explainability & Visualization System

**GAT + InfoNCE 모델** 기반 특허 소송 예측 설명 시각화

```
┌─────────────────────────────────────────────────────────┐
│  1. Intelligence Layer  — HeteroGAT + GAT Attention     │
│  2. Communication Layer — FastAPI REST                  │
│  3. Visualization UI    — Streamlit Dark Dashboard      │
│  4. Graph Engine        — PyVis (Force-Directed, Neon)  │
└─────────────────────────────────────────────────────────┘
```

### 실행 순서
1. `gnn_266_v2.ipynb` Cell 1~12 실행 후 **Cell S (Save Checkpoint)** 실행
2. 이 노트북 전체 실행
3. Cell 12에서 FastAPI 서버 기동 → Cell 13에서 Streamlit 기동

In [ ]:
# Cell 01 — Imports & Package Check
import subprocess, sys, re, warnings, json, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import HTML, display
from sklearn.metrics import roc_auc_score

import faiss
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv, SAGEConv

# ── 필수 패키지 자동 설치 ──────────────────────────────────────────────
REQUIRED_PKGS = {
    'pyvis':                 'pyvis',
    'fastapi':               'fastapi',
    'uvicorn':               'uvicorn[standard]',
    'streamlit':             'streamlit',
    'streamlit_option_menu': 'streamlit-option-menu',
    'streamlit_extras':      'streamlit-extras',
}
for import_name, install_name in REQUIRED_PKGS.items():
    try:
        __import__(import_name)
        print(f'  {import_name}: OK')
    except ImportError:
        print(f'  {import_name}: installing {install_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', install_name, '-q'])
        print(f'  {import_name}: installed')

from pyvis.network import Network

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}  |  PyG: {torch_geometric.__version__}  |  PyTorch: {torch.__version__}')
print(f'NetworkX: {nx.__version__}')

In [ ]:
# Cell 02 — Paths
ROOT     = Path('../..').resolve()
SRC      = ROOT / 'src'
DATA     = ROOT / 'data' / 'samples'
VIZ_DIR  = Path('.').resolve()          # src/data_266/
CKPT     = VIZ_DIR / 'gat_infonce_ckpt.pt'
DATA_PT  = VIZ_DIR / 'hetero_data.pt'
OUT_HTML = VIZ_DIR / 'explain_graph.html'

FAISS_INDEX = Path('/home2/jmbyeon/developments/scratch/patent_eval/uspto_db/data/embeddings/faiss.index')
PATENT_IDX  = Path('/home2/jmbyeon/developments/scratch/patent_eval/uspto_db/data/embeddings/patent_index.parquet')

print('ROOT :', ROOT)
print('CKPT :', CKPT.exists(), CKPT)
print('DATA :', DATA_PT.exists(), DATA_PT)

### ⚠️ gnn_266_v2.ipynb에 아래 셀을 추가하고 실행하세요 (최초 1회)

Cell 12 바로 다음에 붙여넣기:

```python
# Cell S — Save Checkpoint for Visualization
from pathlib import Path
VIZ_DIR = ROOT / 'src' / 'data_266'

# GAT+InfoNCE best state는 Cell 12 마지막 실험(GAT+InfoNCE) 후 model에 로드되어 있음
torch.save({
    'model_state': {k: v.cpu() for k, v in model.state_dict().items()},
    'pat_list':    pat_list,
    'co_list':     co_list,
    'ipc_list':    ipc_list,
    'pat_emb':     pat_emb,
    'test_edges':  test_edges,
    'test_cases':  list(test_cases),
    'sued_edges':  sued_edges,
    'sued_cases':  sued_cases,
    'pat_to_idx':  pat_to_idx,
    'idx_to_pid':  idx_to_pid,
    'co_to_idx':   co_to_idx,
    'ipc_to_idx':  ipc_to_idx,
    'df_ipc':      df_ipc,
    'df_cit':      df_cit,
    'df_asgn':     df_asgn,
}, VIZ_DIR / 'gat_infonce_ckpt.pt')
torch.save(data.cpu(), VIZ_DIR / 'hetero_data.pt')
print('Checkpoint saved →', VIZ_DIR)
```

In [ ]:
# Cell 03 — Model Definition (HeteroGAT, identical to gnn_266_v2)
HIDDEN_DIM = 256
OUT_DIM    = 128
N_LAYERS   = 3
HEADS      = 8
HEAD_DIM   = HIDDEN_DIM // HEADS
FAISS_DIM  = 1024

EDGE_TYPES = [
    ('patent',  'cites',        'patent'),
    ('patent',  'cited_by',     'patent'),
    ('patent',  'has_ipc',      'ipc'),
    ('ipc',     'rev_has_ipc',  'patent'),
    ('company', 'owns',         'patent'),
    ('patent',  'owned_by',     'company'),
]

class _HeteroGNNBase(nn.Module):
    def __init__(self, pat_dim, n_company, n_ipc, hidden, out, n_layers, drop):
        super().__init__()
        self.pat_proj = nn.Sequential(nn.Linear(pat_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.co_emb   = nn.Embedding(n_company, hidden)
        self.ipc_emb  = nn.Embedding(n_ipc,     hidden)
        self.out_proj = nn.Linear(hidden, out)
        self.drop     = nn.Dropout(drop)
        self.norms    = nn.ModuleList([
            nn.ModuleDict({nt: nn.LayerNorm(hidden) for nt in ['patent','company','ipc']})
            for _ in range(n_layers)
        ])

    def forward(self, hetero_data):
        x = {
            'patent':  self.pat_proj(hetero_data['patent'].x),
            'company': self.co_emb.weight,
            'ipc':     self.ipc_emb.weight,
        }
        for conv, norm in zip(self.convs, self.norms):
            x_new = conv(x, hetero_data.edge_index_dict)
            x = {
                k: self.drop(F.relu(norm[k](x_new[k] + x[k]))) if k in norm
                else self.drop(F.relu(x_new[k]))
                for k in x_new
            }
        return {k: self.out_proj(v) for k, v in x.items()}


class HeteroGAT(_HeteroGNNBase):
    def __init__(self, pat_dim, n_company, n_ipc,
                 hidden=HIDDEN_DIM, out=OUT_DIM, n_layers=N_LAYERS,
                 heads=HEADS, drop=0.2):
        super().__init__(pat_dim, n_company, n_ipc, hidden, out, n_layers, drop)
        head_dim = hidden // heads
        self.convs = nn.ModuleList([
            HeteroConv({
                et: GATConv(hidden, head_dim, heads=heads,
                            dropout=drop, concat=True, add_self_loops=False)
                for et in EDGE_TYPES
            }, aggr='mean')
            for _ in range(n_layers)
        ])

print('HeteroGAT 정의 완료')

In [ ]:
# Cell 04 — Load Checkpoint + Build Model
assert CKPT.exists(), f'체크포인트 없음: {CKPT}\n→ gnn_266_v2.ipynb에 Cell S를 추가하고 실행하세요'

ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)

pat_list   = ckpt['pat_list']
co_list    = ckpt['co_list']
ipc_list   = ckpt['ipc_list']
pat_emb    = ckpt['pat_emb']
test_edges = ckpt['test_edges']
test_cases = set(ckpt['test_cases'])
sued_edges = ckpt['sued_edges']
sued_cases = ckpt['sued_cases']
pat_to_idx = ckpt['pat_to_idx']
idx_to_pid = ckpt['idx_to_pid']
co_to_idx  = ckpt['co_to_idx']
ipc_to_idx = ckpt['ipc_to_idx']
df_ipc     = ckpt['df_ipc']
df_cit     = ckpt['df_cit']
df_asgn    = ckpt['df_asgn']

ipc_by_pat = df_ipc.groupby('patent_id')['ipc_code'].apply(set).to_dict()

data   = torch.load(DATA_PT, map_location='cpu', weights_only=False)
data_d = data.to(device)

model = HeteroGAT(FAISS_DIM, len(co_list), len(ipc_list)).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Patents  : {len(pat_list)}')
print(f'Companies: {len(co_list)}')
print(f'IPC codes: {len(ipc_list)}')
print(f'Test pairs: {test_edges.shape[1]}')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# Sanity check
with torch.no_grad():
    h = model(data_d)['patent']
    h_norm = F.normalize(h, dim=1)
    pos_sim = F.cosine_similarity(h_norm[test_edges[0].to(device)],
                                  h_norm[test_edges[1].to(device)]).mean()
print(f'\nSanity check — test pos cosine mean: {pos_sim:.4f}  (expected ~0.78)')

---
## Section 1 — GAT Attention Weight 추출

HeteroGAT의 각 레이어 × 각 엣지 타입별 attention α를 직접 추출합니다.
- `α`: 각 (src→dst) 엣지가 dst 노드 표현 계산에 기여한 가중치
- 8 heads의 평균값 사용

In [ ]:
# Cell 05 — GAT Attention Extraction Function
@torch.no_grad()
def extract_gat_attention(model, hetero_data):
    """
    레이어별 · 엣지타입별 GAT attention weight 추출.
    Returns: list[dict]  — layer 수 길이, 각 항목:
        { edge_type_tuple: {'edge_index': (2,E), 'alpha': (E,)} }
    """
    model.eval()
    d = hetero_data.to(next(model.parameters()).device)

    x = {
        'patent':  model.pat_proj(d['patent'].x),
        'company': model.co_emb.weight,
        'ipc':     model.ipc_emb.weight,
    }

    all_layers = []

    for hetero_conv, norm in zip(model.convs, model.norms):
        layer_attn = {}

        # HeteroConv.convs 키 형식: 'src__rel__dst'
        for et_str, gat_conv in hetero_conv.convs.items():
            parts = et_str.split('__')
            et = (parts[0], parts[1], parts[2])
            if et not in d.edge_index_dict:
                continue

            src_type, _, dst_type = et
            ei = d.edge_index_dict[et]

            _, (ei_ret, alpha) = gat_conv(
                (x[src_type], x[dst_type]),
                ei,
                return_attention_weights=True
            )
            layer_attn[et] = {
                'edge_index': ei_ret.cpu(),
                'alpha':      alpha.mean(dim=-1).cpu(),   # (E,)  heads 평균
                'alpha_heads': alpha.cpu(),               # (E, heads)
            }

        # 다음 레이어를 위한 x 업데이트 (원본 forward 로직)
        x_new = hetero_conv(x, d.edge_index_dict)
        x = {
            k: model.drop(F.relu(norm[k](x_new[k] + x[k]))) if k in norm
            else model.drop(F.relu(x_new[k]))
            for k in x_new
        }
        all_layers.append(layer_attn)

    return all_layers


print('extract_gat_attention 정의 완료')

In [ ]:
# Cell 06 — Attention 추출 실행 + 통계
attn_layers = extract_gat_attention(model, data_d)

print(f'추출 레이어 수: {len(attn_layers)}')
for li, layer in enumerate(attn_layers):
    print(f'\n  Layer {li+1}:')
    for et, attn in layer.items():
        a = attn['alpha']
        print(f'    {str(et):<45}  E={len(a):4d}  '
              f'mean={a.mean():.4f}  max={a.max():.4f}  std={a.std():.4f}')

In [ ]:
# Cell 07 — Attention Heatmap (엣지 타입 × 레이어)
ET_LABELS = {
    ('patent',  'cites',        'patent'):  'pat→cites→pat',
    ('patent',  'cited_by',     'patent'):  'pat→cited_by→pat',
    ('patent',  'has_ipc',      'ipc'):     'pat→has_ipc→ipc',
    ('ipc',     'rev_has_ipc',  'patent'):  'ipc→rev_ipc→pat',
    ('company', 'owns',         'patent'):  'co→owns→pat',
    ('patent',  'owned_by',     'company'): 'pat→owned_by→co',
}
ET_COLORS = {
    ('patent',  'cites',        'patent'):  '#00BFFF',
    ('patent',  'cited_by',     'patent'):  '#1E90FF',
    ('patent',  'has_ipc',      'ipc'):     '#7FFF00',
    ('ipc',     'rev_has_ipc',  'patent'):  '#32CD32',
    ('company', 'owns',         'patent'):  '#FF69B4',
    ('patent',  'owned_by',     'company'): '#DA70D6',
}

n_layers = len(attn_layers)
all_ets  = list(ET_LABELS.keys())

fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 5),
                         facecolor='#0E1117')
if n_layers == 1:
    axes = [axes]

for li, (ax, layer) in enumerate(zip(axes, attn_layers)):
    means  = [layer[et]['alpha'].mean().item() if et in layer else 0 for et in all_ets]
    colors = [ET_COLORS.get(et, '#888888') for et in all_ets]
    labels = [ET_LABELS.get(et, str(et)) for et in all_ets]

    bars = ax.barh(labels, means, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, means):
        ax.text(v + 0.0005, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', color='white', fontsize=8)

    ax.set_facecolor('#0E1117')
    ax.tick_params(colors='white')
    ax.set_xlabel('Mean Attention α', color='white')
    ax.set_title(f'Layer {li+1}', color='#00FFFF', fontsize=12, pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

fig.suptitle('GAT Attention — Mean α per Edge Type per Layer',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(VIZ_DIR / 'attention_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='#0E1117')
plt.show()
print('저장:', VIZ_DIR / 'attention_heatmap.png')

---
## Section 2 — 특허 쌍 설명 서브그래프 추출

원고 특허 → 피고 특허 예측에 기여한 주변 구조를 추출합니다.
- 마지막 레이어(가장 정제된) attention 기준
- 두 특허와 연결된 상위 K개 엣지 선택
- Patent / Company / IPC 노드 타입 보존

In [ ]:
# Cell 08 — Subgraph Extraction for a Patent Pair
def get_pair_explanation(attn_layers, src_pat_idx, dst_pat_idx,
                         top_k_per_type=8):
    """
    (원고특허 src, 피고특허 dst) 쌍에 대한 설명 서브그래프 추출.

    Returns:
        nodes: dict  {node_type: set of node_ids}
        edges: list  [{src_type, src_id, dst_type, dst_id, rel, weight}]
    """
    target_pats = {src_pat_idx, dst_pat_idx}
    nodes = {'patent': set(target_pats), 'company': set(), 'ipc': set()}
    edges = []
    seen_edges = set()

    # 마지막 레이어 attention 사용 (가장 정제됨)
    last_layer = attn_layers[-1]

    for et, attn_data in last_layer.items():
        src_type, rel, dst_type = et
        ei    = attn_data['edge_index']   # (2, E)
        alpha = attn_data['alpha']         # (E,)

        # 두 타겟 특허와 연결된 엣지 필터
        if src_type == 'patent':
            mask = torch.isin(ei[0], torch.tensor(list(target_pats)))
        elif dst_type == 'patent':
            mask = torch.isin(ei[1], torch.tensor(list(target_pats)))
        else:
            continue

        if mask.sum() == 0:
            continue

        rel_ei    = ei[:, mask]
        rel_alpha = alpha[mask]
        k = min(top_k_per_type, mask.sum().item())
        top_idx   = rel_alpha.topk(k).indices

        for idx in top_idx:
            s = int(rel_ei[0, idx])
            d = int(rel_ei[1, idx])
            w = float(rel_alpha[idx])
            key = (src_type, s, dst_type, d, rel)
            if key in seen_edges:
                continue
            seen_edges.add(key)

            edges.append({'src_type': src_type, 'src_id': s,
                          'dst_type': dst_type, 'dst_id': d,
                          'rel': rel, 'weight': w})
            nodes[src_type].add(s)
            nodes[dst_type].add(d)

    # 노드 수 출력
    total_nodes = sum(len(v) for v in nodes.values())
    print(f'서브그래프: {total_nodes} 노드 / {len(edges)} 엣지')
    for k, v in nodes.items():
        print(f'  {k}: {len(v)}')
    return nodes, edges


print('get_pair_explanation 정의 완료')

---
## Section 3 — PyVis 네온 다크 그래프 엔진

```
Patent  ●  Cyan   #00FFFF
Company ●  Magenta #FF00FF
IPC     ●  Lime   #00FF41

cites / cited_by  ──  Blue glow
has_ipc / rev     ──  Green glow
owns / owned_by   ──  Pink glow

엣지 굵기 = attention 가중치 비례
```

In [ ]:
# Cell 09 — PyVis Dark Neon Graph Builder
NODE_COLOR = {'patent': '#00FFFF', 'company': '#FF00FF', 'ipc': '#00FF41'}
NODE_SIZE  = {'patent': 28, 'company': 20, 'ipc': 16}
EDGE_COLOR = {
    'cites':       '#00BFFF',
    'cited_by':    '#1E90FF',
    'has_ipc':     '#7FFF00',
    'rev_has_ipc': '#32CD32',
    'owns':        '#FF69B4',
    'owned_by':    '#DA70D6',
}
HIGHLIGHT_SRC = '#FFD700'   # Gold — 원고특허
HIGHLIGHT_DST = '#FF4500'   # OrangeRed — 피고특허


def _node_label(ntype, nid, pat_list, co_list, ipc_list, max_len=18):
    if ntype == 'patent':
        return str(pat_list[nid])
    elif ntype == 'company':
        s = str(co_list[nid])
        return s[:max_len] + '..' if len(s) > max_len else s
    else:
        return str(ipc_list[nid])


def build_neon_graph(nodes, edges, src_pat_idx, dst_pat_idx,
                     pat_list, co_list, ipc_list,
                     cosine_score=None,
                     output_html=None,
                     height='720px'):
    """
    PyVis 기반 네온 다크 설명 그래프 생성.
    Returns: pyvis.Network (notebook 렌더링 가능)
    """
    net = Network(
        height=height, width='100%',
        bgcolor='#0E1117', font_color='#FFFFFFCC',
        directed=True,
        notebook=True,
        cdn_resources='in_line',
    )

    def uid(ntype, nid):
        return f'{ntype[0]}_{nid}'

    # ── Nodes ──
    added = set()
    for ntype, nids in nodes.items():
        for nid in nids:
            nuid = uid(ntype, nid)
            if nuid in added:
                continue
            added.add(nuid)

            is_src = (ntype == 'patent' and nid == src_pat_idx)
            is_dst = (ntype == 'patent' and nid == dst_pat_idx)
            label  = _node_label(ntype, nid, pat_list, co_list, ipc_list)

            if is_src:
                color, size = HIGHLIGHT_SRC, 40
                tag = ' ★ PLAINTIFF'
            elif is_dst:
                color, size = HIGHLIGHT_DST, 40
                tag = ' ★ DEFENDANT'
            else:
                color = NODE_COLOR[ntype]
                size  = NODE_SIZE[ntype]
                tag   = ''

            title = (f'<b>[{ntype.upper()}]</b><br>'
                     f'{label}{tag}')
            if is_src or is_dst:
                score_txt = f'<br>Cosine: {cosine_score:.4f}' if cosine_score else ''
                title += score_txt

            net.add_node(
                nuid, label=label, title=title,
                color={
                    'background': color,
                    'border': '#FFFFFF' if (is_src or is_dst) else color,
                    'highlight': {'background': '#FFFFFF', 'border': '#FFFFFF'},
                    'hover':     {'background': '#FFFFFF', 'border': '#FFFFFF'},
                },
                size=size,
                font={'color': '#FFFFFF', 'size': 10, 'bold': is_src or is_dst},
                borderWidth=3 if (is_src or is_dst) else 1,
                shadow={'enabled': True,
                        'color': color + '88',
                        'size': 20 if (is_src or is_dst) else 10,
                        'x': 0, 'y': 0},
            )

    # ── Edges ──
    max_w = max((e['weight'] for e in edges), default=1.0)

    for e in edges:
        su = uid(e['src_type'], e['src_id'])
        du = uid(e['dst_type'], e['dst_id'])
        if su not in added or du not in added:
            continue

        norm_w = e['weight'] / (max_w + 1e-8)
        width  = 1.0 + norm_w * 9.0    # 1 ~ 10
        ec     = EDGE_COLOR.get(e['rel'], '#888888')

        net.add_edge(
            su, du,
            title=f"<b>{e['rel']}</b><br>α = {e['weight']:.4f}",
            width=width,
            color={'color': ec, 'highlight': '#FFFFFF', 'hover': '#FFFFFF'},
            arrows={'to': {'enabled': True, 'scaleFactor': 0.6}},
            smooth={'type': 'curvedCW', 'roundness': 0.2},
            shadow={'enabled': True, 'color': ec + 'AA', 'size': 8, 'x': 0, 'y': 0},
        )

    # ── Physics (Force-Directed) ──
    net.set_options(json.dumps({
        'physics': {
            'enabled': True,
            'solver': 'forceAtlas2Based',
            'forceAtlas2Based': {
                'gravitationalConstant': -60,
                'centralGravity': 0.005,
                'springLength': 140,
                'springConstant': 0.06,
                'damping': 0.35,
                'avoidOverlap': 0.8,
            },
            'stabilization': {'iterations': 200, 'updateInterval': 50},
        },
        'interaction': {
            'hover': True,
            'tooltipDelay': 100,
            'hideEdgesOnDrag': True,
        },
        'nodes': {'shape': 'dot'},
        'edges': {'smooth': {'type': 'curvedCW', 'roundness': 0.2}},
    }))

    if output_html:
        net.save_graph(str(output_html))
        print(f'HTML 저장 → {output_html}')

    return net


print('build_neon_graph 정의 완료')

In [ ]:
# Cell 10 — Demo: test셋 특허 쌍 시각화
#
# 원하는 쌍 선택 방법:
#   A) test_edges에서 직접 인덱스 지정
#   B) 케이스 ID로 필터
#   C) 특허 번호(patent_id)로 직접 지정

# ── 쌍 선택 ──
PAIR_IDX = 0    # test_edges에서 몇 번째 쌍을 볼지 (0~164)

src_idx = int(test_edges[0, PAIR_IDX])
dst_idx = int(test_edges[1, PAIR_IDX])
src_pid = pat_list[src_idx]
dst_pid = pat_list[dst_idx]

# 예측 점수
with torch.no_grad():
    h    = F.normalize(model(data_d)['patent'], dim=1)
    cos  = F.cosine_similarity(
        h[src_idx].unsqueeze(0), h[dst_idx].unsqueeze(0)
    ).item()

print(f'원고 특허: {src_pid}  (idx={src_idx})')
print(f'피고 특허: {dst_pid}  (idx={dst_idx})')
print(f'Cosine score: {cos:.4f}')
print(f'IPC(원고): {ipc_by_pat.get(src_pid, set())}')
print(f'IPC(피고): {ipc_by_pat.get(dst_pid, set())}')

# ── 설명 서브그래프 추출 ──
nodes, edges = get_pair_explanation(attn_layers, src_idx, dst_idx, top_k_per_type=8)

# ── PyVis 그래프 생성 ──
net = build_neon_graph(
    nodes, edges, src_idx, dst_idx,
    pat_list, co_list, ipc_list,
    cosine_score=cos,
    output_html=OUT_HTML,
)

# ── Jupyter inline 렌더링 ──
display(HTML(net.generate_html()))

In [ ]:
# Cell 11 — 특허 ID로 Top-K 피고 특허 예측 + 시각화

def predict_top_k(query_pid, model, data_d, pat_list, pat_to_idx, k=10):
    """
    query_pid: 특허 번호 문자열 (예: '7123456')
    Returns: DataFrame [patent_id, cosine_score, rank]
    """
    if query_pid not in pat_to_idx:
        raise ValueError(f'{query_pid} not in patent list')

    q_idx = pat_to_idx[query_pid]
    with torch.no_grad():
        h = F.normalize(model(data_d)['patent'], dim=1).cpu().numpy()

    sims = h @ h[q_idx]
    sims[q_idx] = -np.inf   # 자기 자신 제외
    top_idx = np.argsort(-sims)[:k]

    return pd.DataFrame({
        'patent_id':    [pat_list[i] for i in top_idx],
        'node_idx':     top_idx,
        'cosine_score': sims[top_idx],
        'rank':         range(1, k+1),
    })


# Demo: test_edges의 첫 번째 원고 특허 기준 Top-10
query_pid = src_pid
df_topk   = predict_top_k(query_pid, model, data_d, pat_list, pat_to_idx, k=10)

print(f'Query: {query_pid}  →  Top-10 predicted defendants')
print(df_topk.to_string(index=False))
print(f'\n실제 피고 특허 {dst_pid}의 순위: '
      f"{df_topk[df_topk['patent_id']==dst_pid]['rank'].values}")

In [ ]:
# Cell 12 — Gradient-based Edge Importance (GNNExplainer 스타일)
#
# 원리: cosine score의 각 엣지 src·dst 노드 특성에 대한 gradient 크기
#        → 해당 엣지가 예측에 미치는 민감도

def gradient_edge_importance(model, data_d, src_idx, dst_idx, layer_idx=-1):
    """
    특정 특허 쌍 (src, dst)에 대해 각 엣지 타입별 중요도를 gradient로 산출.
    layer_idx: 어떤 레이어의 gradient를 볼지 (-1 = 마지막)
    """
    model.eval()
    d = data_d

    # x를 requires_grad로 계산
    x = {
        'patent':  model.pat_proj(d['patent'].x),
        'company': model.co_emb.weight.detach().clone().requires_grad_(True),
        'ipc':     model.ipc_emb.weight.detach().clone().requires_grad_(True),
    }
    x['patent'] = x['patent'].detach().clone().requires_grad_(True)

    # Forward
    for conv, norm in zip(model.convs, model.norms):
        x_new = conv(x, d.edge_index_dict)
        x = {
            k: F.relu(norm[k](x_new[k] + x[k])) if k in norm
            else F.relu(x_new[k])
            for k in x_new
        }

    h = model.out_proj(x['patent'])
    h = F.normalize(h, dim=1)

    score = F.cosine_similarity(h[src_idx].unsqueeze(0), h[dst_idx].unsqueeze(0))
    score.backward()

    # 엣지별 중요도: 엣지 dst 노드의 gradient norm
    importance = {}
    for et, attn_data in attn_layers[-1].items():
        src_type, rel, dst_type = et
        if dst_type != 'patent':
            continue
        ei    = attn_data['edge_index'].to(device)
        alpha = attn_data['alpha'].to(device)

        # dst 노드의 gradient (patent만)
        grad = x['patent'].grad  # (N_patent, hidden)
        if grad is None:
            continue

        dst_nodes = ei[1]                            # (E,)
        dst_grads = grad[dst_nodes].norm(dim=1)      # (E,)

        # Attention × Gradient  (element-wise)
        combined = (alpha * dst_grads).cpu()
        importance[et] = {
            'edge_index': ei.cpu(),
            'score':      combined,
        }

    return importance


# Demo
imp = gradient_edge_importance(model, data_d, src_idx, dst_idx)

print(f'Gradient × Attention Edge Importance ({src_pid} → {dst_pid})')
print(f'{"Edge Type":<45} {"Max Score":>10} {"Mean Score":>10}')
print('-' * 68)
for et, v in imp.items():
    s = v['score']
    print(f'{str(et):<45} {s.max():>10.4f} {s.mean():>10.4f}')

---
## Section 4 — 갤럭시 그래프 빌더 (Train/Val 전용)

Test 노드 대기실 구조의 핵심:
- **Galaxy** = Train+Val 특허 (기존 소송 네트워크 — 고정)
- **Waiting Room** = Test 특허 (아직 분석 안 된 신규 후보군)
- **Injection** = 클릭 → 신규 노드가 갤럭시로 낙하 + 예측 엣지 발사

In [ ]:
# Cell 16 — Train/Val vs Test 특허 분리
#
# Galaxy  = train+val 특허 (기존 소송 네트워크, 고정)
# Waiting = test 특허   (대기실, 미분석)

import numpy as np

sued_edges_np = sued_edges.numpy() if hasattr(sued_edges, 'numpy') else np.array(sued_edges)

# Test 케이스에 속하는 쌍만 마스킹
test_case_mask = np.array([c in test_cases for c in sued_cases])

test_src_set = set(sued_edges_np[0, test_case_mask].tolist())
test_dst_set = set(sued_edges_np[1, test_case_mask].tolist())
test_pat_set = test_src_set | test_dst_set       # 대기실 노드들

all_pat_set    = set(range(len(pat_list)))
galaxy_pat_set = all_pat_set - test_pat_set       # 갤럭시 노드들

# Train+Val 소송 역할
tv_mask       = ~test_case_mask
tv_plt_set    = set(sued_edges_np[0, tv_mask].tolist())   # 원고
tv_def_set    = set(sued_edges_np[1, tv_mask].tolist())   # 피고
tv_both_set   = tv_plt_set & tv_def_set                   # 양쪽 다

# Test 쌍 정보 (대기실 버튼용)
test_pairs_info = {}   # test_pat_idx → [{'pair_idx': dst_idx, 'case': case_id}]
for i, (c, in_test) in enumerate(zip(sued_cases, test_case_mask)):
    if not in_test: continue
    s = int(sued_edges_np[0, i])
    d = int(sued_edges_np[1, i])
    test_pairs_info.setdefault(s, []).append({'pair': d, 'case': c})
    test_pairs_info.setdefault(d, []).append({'pair': s, 'case': c})

print(f'Galaxy  (train+val) : {len(galaxy_pat_set):4d} 특허')
print(f'Waiting (test)      : {len(test_pat_set):4d} 특허')
print(f'  — plaintiff only  : {len(test_src_set - test_dst_set)}')
print(f'  — defendant only  : {len(test_dst_set - test_src_set)}')
print(f'  — both sides      : {len(test_src_set & test_dst_set)}')
print(f'Galaxy 내 litigated : {len(tv_plt_set | tv_def_set)} (원고 {len(tv_plt_set)}, 피고 {len(tv_def_set)})')

In [ ]:
# Cell 17 — Galaxy Graph Builder (Train+Val 네트워크)
GALAXY_OPTS = json.dumps({
    "physics": {
        "solver": "forceAtlas2Based",
        "forceAtlas2Based": {
            "gravitationalConstant": -55,
            "centralGravity": 0.003,
            "springLength": 120,
            "springConstant": 0.04,
            "damping": 0.42,
            "avoidOverlap": 0.6,
        },
        "stabilization": {"iterations": 300, "updateInterval": 80, "fit": True},
        "maxVelocity": 60,
        "minVelocity": 0.8,
    },
    "interaction": {
        "hover": True, "tooltipDelay": 80,
        "hideEdgesOnDrag": True, "navigationButtons": True,
    },
    "nodes": {"shape": "dot"},
    "edges": {"smooth": {"type": "continuous"}},
})


def build_galaxy_graph(
    galaxy_pat_set, tv_plt_set, tv_def_set,
    pat_list, co_list, ipc_list,
    df_cit, df_ipc, df_asgn, pat_to_idx, ipc_to_idx, co_to_idx,
    test_pat_idx=None,           # 주입할 테스트 노드 (선택)
    predicted_edges=None,        # [(dst_idx, cosine_score), ...]
    height="820px",
):
    """
    기존 소송 생태계(train+val)를 '은하수' 형태로 시각화.
    test_pat_idx가 주어지면 해당 노드를 갤럭시에 주입하고 예측 엣지를 그린다.
    """
    net = Network(
        height=height, width="100%",
        bgcolor="#050810", font_color="#FFFFFF",
        directed=True, notebook=False, cdn_resources="in_line",
    )

    # ── 1. 특허 노드 (Galaxy) ──────────────────
    litigated = tv_plt_set | tv_def_set
    for idx in galaxy_pat_set:
        pid = str(pat_list[idx])
        in_both = idx in tv_plt_set and idx in tv_def_set
        in_plt  = idx in tv_plt_set
        in_def  = idx in tv_def_set

        if in_both:
            color = "#FFA500"; role = "P&D"; sz = 26
        elif in_plt:
            color = "#FFD700"; role = "Plaintiff"; sz = 22
        elif in_def:
            color = "#FF6B35"; role = "Defendant"; sz = 22
        else:
            color = "#007B9E"; role = "Connected"; sz = 12  # 희미한 시안

        shadow_size = 14 if idx in litigated else 5
        net.add_node(
            f"p_{idx}", label=pid,
            title=(f"<b>[PATENT]</b><br>{pid}<br>"
                   f"<span style='color:{color}'>◉ {role}</span>"),
            color={
                "background": color,
                "border": color,
                "highlight": {"background": "#FFFFFF", "border": "#FFFFFF"},
                "hover":     {"background": "#FFFFFF", "border": "#FFFFFF"},
            },
            size=sz,
            font={"color": "#AAAAAA" if role == "Connected" else "#FFFFFF",
                  "size": 9 if role == "Connected" else 11},
            borderWidth=1,
            shadow={"enabled": True, "color": color + "55",
                    "size": shadow_size, "x": 0, "y": 0},
        )

    # ── 2. Citation 엣지 (galaxy 내부) ──────────
    added_cite = set()
    for _, row in df_cit.iterrows():
        si = pat_to_idx.get(str(row.get('citing_patent', row.get('patent_id', ''))))
        di = pat_to_idx.get(str(row.get('cited_patent', row.get('cited_id', ''))))
        if si is None or di is None: continue
        if si not in galaxy_pat_set or di not in galaxy_pat_set: continue
        key = (si, di)
        if key in added_cite: continue
        added_cite.add(key)
        net.add_edge(
            f"p_{si}", f"p_{di}",
            color={"color": "#003B5C", "hover": "#00BFFF"},
            width=1.0, arrows={"to": {"enabled": True, "scaleFactor": 0.4}},
            smooth={"type": "continuous"},
        )

    # ── 3. IPC 노드 + 엣지 (소송특허 연결만) ────
    ipc_in_graph = {}
    for _, row in df_ipc.iterrows():
        pid_str = str(row.get('patent_id', ''))
        pi = pat_to_idx.get(pid_str)
        ic_str = str(row.get('ipc_code', ''))
        ci = ipc_to_idx.get(ic_str)
        if pi is None or ci is None: continue
        if pi not in litigated: continue        # 소송 특허만 IPC 표시
        if pi not in galaxy_pat_set: continue
        ipc_in_graph[ci] = ic_str
        net.add_edge(
            f"p_{pi}", f"i_{ci}",
            color={"color": "#1A3300", "hover": "#7FFF00"},
            width=0.8, arrows={"to": {"enabled": False}},
            smooth={"type": "continuous"},
        )

    for ci, ic_str in ipc_in_graph.items():
        net.add_node(
            f"i_{ci}", label=ic_str,
            title=f"<b>[IPC]</b><br>{ic_str}",
            color={"background": "#1A4400", "border": "#32CD32",
                   "hover": {"background": "#32CD32"}},
            size=11, shape="diamond",
            font={"color": "#32CD32", "size": 9},
        )

    # ── 4. Company 노드 + 엣지 (소송특허 연결만) ─
    co_in_graph = {}
    for _, row in df_asgn.iterrows():
        pid_str = str(row.get('patent_id', ''))
        pi = pat_to_idx.get(pid_str)
        co_str = str(row.get('assignee', row.get('company', '')))
        ci2 = co_to_idx.get(co_str)
        if pi is None or ci2 is None: continue
        if pi not in litigated: continue
        if pi not in galaxy_pat_set: continue
        co_in_graph[ci2] = co_str
        net.add_edge(
            f"co_{ci2}", f"p_{pi}",
            color={"color": "#2A0030", "hover": "#FF69B4"},
            width=0.8, arrows={"to": {"enabled": True, "scaleFactor": 0.3}},
            smooth={"type": "continuous"},
        )

    for ci2, co_str in co_in_graph.items():
        label_co = co_str[:16] + ".." if len(co_str) > 16 else co_str
        net.add_node(
            f"co_{ci2}", label=label_co,
            title=f"<b>[COMPANY]</b><br>{co_str}",
            color={"background": "#2A0040", "border": "#FF00FF",
                   "hover": {"background": "#FF00FF"}},
            size=14, shape="square",
            font={"color": "#FF00FF", "size": 9},
        )

    # ── 5. Test 노드 주입 (선택) ─────────────────
    if test_pat_idx is not None:
        tpid = str(pat_list[test_pat_idx])
        net.add_node(
            f"p_{test_pat_idx}", label=tpid,
            title=(f"<b>[TEST — INCOMING]</b><br>{tpid}<br>"
                   f"<span style='color:#FFFFFF'>🛸 방금 주입된 신규 노드</span>"),
            color={
                "background": "#FFFFFF",
                "border": "#00FFFF",
                "highlight": {"background": "#00FFFF", "border": "#FFFFFF"},
            },
            size=48,
            font={"color": "#FFFFFF", "size": 13, "bold": True},
            borderWidth=4,
            shadow={"enabled": True, "color": "#FFFFFF99", "size": 36, "x": 0, "y": 0},
        )

        # 예측 엣지 (Neon Pink — 소송 위험도)
        if predicted_edges:
            max_cos = max(c for _, c in predicted_edges)
            for rank, (dst_idx, cos_score) in enumerate(predicted_edges, 1):
                if f"p_{dst_idx}" not in {n['id'] for n in net.nodes}: continue
                norm = cos_score / (max_cos + 1e-8)
                width = 2 + norm * 12          # 2 ~ 14
                danger_color = (
                    "#FF0066" if cos_score > 0.75 else
                    "#FF6600" if cos_score > 0.55 else
                    "#FFAA00"
                )
                net.add_edge(
                    f"p_{test_pat_idx}",
                    f"p_{dst_idx}",
                    label=f"#{rank}  {cos_score:.3f}",
                    font={"color": danger_color, "size": 10,
                          "bold": True, "align": "middle"},
                    title=(f"<b>⚠️ 소송 예측</b><br>"
                           f"Cosine: {cos_score:.4f}<br>Rank: {rank}"),
                    width=width,
                    color={"color": danger_color,
                           "highlight": "#FFFFFF", "hover": "#FFFFFF"},
                    arrows={"to": {"enabled": True, "scaleFactor": 0.8}},
                    smooth={"type": "curvedCW", "roundness": 0.3},
                    shadow={"enabled": True,
                            "color": danger_color + "CC",
                            "size": 16, "x": 0, "y": 0},
                    dashes=False,
                )

    net.set_options(GALAXY_OPTS)
    return net


print('build_galaxy_graph 정의 완료')
print(f'  Galaxy 노드 예정: {len(galaxy_pat_set)} patents')

In [ ]:
# Cell 18 — Galaxy + Test Node Injection 데모
#
# 시나리오: test_pat_set에서 한 특허를 "대기실에서 선택"하여 갤럭시에 주입

# ── 1. 테스트 노드 선택 ──────────────────────
TEST_PAT_IDX = int(test_edges[0, 0].item())   # test_edges 첫 원고특허
test_pid     = pat_list[TEST_PAT_IDX]
print(f'선택된 Test 특허: {test_pid}  (idx={TEST_PAT_IDX})')

# ── 2. Top-K 예측 ────────────────────────────
with torch.no_grad():
    h_norm = F.normalize(model(data_d)['patent'], dim=1).cpu().numpy()

sims = h_norm @ h_norm[TEST_PAT_IDX]
sims[TEST_PAT_IDX] = -np.inf
# 갤럭시 내 노드만 후보 (test_pat_set 제외)
for ti in test_pat_set:
    sims[ti] = -np.inf

K = 7
top_idx  = np.argsort(-sims)[:K]
pred_edges = [(int(i), float(sims[i])) for i in top_idx]

print(f'\nTop-{K} 예측 피고 (갤럭시 내):')
for r, (di, sc) in enumerate(pred_edges, 1):
    risk = "HIGH" if sc > 0.75 else "MED" if sc > 0.55 else "LOW"
    print(f'  #{r}  {pat_list[di]:<15}  cos={sc:.4f}  [{risk}]')

# ── 3. Galaxy + 주입 그래프 생성 ────────────
galaxy_net = build_galaxy_graph(
    galaxy_pat_set, tv_plt_set, tv_def_set,
    pat_list, co_list, ipc_list,
    df_cit, df_ipc, df_asgn, pat_to_idx, ipc_to_idx, co_to_idx,
    test_pat_idx=TEST_PAT_IDX,
    predicted_edges=pred_edges,
    height="780px",
)

GALAXY_HTML = VIZ_DIR / 'galaxy_injected.html'
galaxy_net.save_graph(str(GALAXY_HTML))
print(f'\n갤럭시 그래프 저장 → {GALAXY_HTML}')
print(f'노드 수: {len(galaxy_net.nodes):,}  엣지 수: {len(galaxy_net.edges):,}')

# Jupyter 인라인 렌더링
display(HTML(galaxy_net.generate_html()))

In [ ]:
# Cell 13 — FastAPI Backend 파일 생성
FASTAPI_CODE = '''
"""app_explain.py — Patent Litigation GNN Explainer API"""
from pathlib import Path
import json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv

warnings.filterwarnings("ignore")
app = FastAPI(title="Patent Litigation Explainer", version="1.0")
app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

# ── Constants ─────────────────────────────────────────────────────────
HIDDEN_DIM, OUT_DIM, N_LAYERS, HEADS = 256, 128, 3, 8
FAISS_DIM = 1024
EDGE_TYPES = [
    ("patent",  "cites",        "patent"),
    ("patent",  "cited_by",     "patent"),
    ("patent",  "has_ipc",      "ipc"),
    ("ipc",     "rev_has_ipc",  "patent"),
    ("company", "owns",         "patent"),
    ("patent",  "owned_by",     "company"),
]
CKPT    = Path("gat_infonce_ckpt.pt")
DATA_PT = Path("hetero_data.pt")

# ── Model ─────────────────────────────────────────────────────────────
class _HeteroGNNBase(nn.Module):
    def __init__(self, pat_dim, n_co, n_ipc, hidden, out, n_layers, drop):
        super().__init__()
        self.pat_proj = nn.Sequential(nn.Linear(pat_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.co_emb  = nn.Embedding(n_co, hidden)
        self.ipc_emb = nn.Embedding(n_ipc, hidden)
        self.out_proj = nn.Linear(hidden, out)
        self.drop = nn.Dropout(drop)
        self.norms = nn.ModuleList([
            nn.ModuleDict({nt: nn.LayerNorm(hidden) for nt in ["patent","company","ipc"]})
            for _ in range(n_layers)
        ])
    def forward(self, data):
        x = {"patent": self.pat_proj(data["patent"].x),
             "company": self.co_emb.weight, "ipc": self.ipc_emb.weight}
        for conv, norm in zip(self.convs, self.norms):
            x_new = conv(x, data.edge_index_dict)
            x = {k: self.drop(F.relu(norm[k](x_new[k]+x[k]))) if k in norm
                 else self.drop(F.relu(x_new[k])) for k in x_new}
        return {k: self.out_proj(v) for k, v in x.items()}

class HeteroGAT(_HeteroGNNBase):
    def __init__(self, pat_dim, n_co, n_ipc,
                 hidden=HIDDEN_DIM, out=OUT_DIM, n_layers=N_LAYERS, heads=HEADS, drop=0.2):
        super().__init__(pat_dim, n_co, n_ipc, hidden, out, n_layers, drop)
        hd = hidden // heads
        self.convs = nn.ModuleList([
            HeteroConv({et: GATConv(hidden, hd, heads=heads,
                        dropout=drop, concat=True, add_self_loops=False)
                       for et in EDGE_TYPES}, aggr="mean")
            for _ in range(n_layers)
        ])

# ── Global State ──────────────────────────────────────────────────────
_state = {}

@app.on_event("startup")
async def startup():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt   = torch.load(CKPT, map_location="cpu", weights_only=False)
    data   = torch.load(DATA_PT, map_location="cpu", weights_only=False)

    model = HeteroGAT(FAISS_DIM, len(ckpt["co_list"]), len(ckpt["ipc_list"])).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    with torch.no_grad():
        h = F.normalize(model(data.to(device))["patent"], dim=1).cpu().numpy()

    _state.update({
        "model": model, "data": data.to(device), "h": h,
        "pat_list": ckpt["pat_list"], "co_list": ckpt["co_list"],
        "ipc_list": ckpt["ipc_list"], "pat_to_idx": ckpt["pat_to_idx"],
        "df_ipc": ckpt["df_ipc"], "device": device,
    })
    print(f"Model loaded on {device}  |  {len(ckpt[\"pat_list\"])} patents")


def _extract_attn(model, data):
    """GAT attention 추출 (마지막 레이어)"""
    model.eval()
    x = {"patent": model.pat_proj(data["patent"].x),
         "company": model.co_emb.weight, "ipc": model.ipc_emb.weight}
    all_attn = []
    for hetero_conv, norm in zip(model.convs, model.norms):
        layer_attn = {}
        for et_str, gat_conv in hetero_conv.convs.items():
            parts = et_str.split("__")
            et = (parts[0], parts[1], parts[2])
            if et not in data.edge_index_dict: continue
            src_t, _, dst_t = et
            ei = data.edge_index_dict[et]
            _, (ei_r, alpha) = gat_conv((x[src_t], x[dst_t]), ei, return_attention_weights=True)
            layer_attn["|".join(et)] = {"edge_index": ei_r.cpu().tolist(),
                                         "alpha": alpha.mean(-1).cpu().tolist()}
        x_new = hetero_conv(x, data.edge_index_dict)
        x = {k: F.relu(norm[k](x_new[k]+x[k])) if k in norm
             else F.relu(x_new[k]) for k in x_new}
        all_attn.append(layer_attn)
    return all_attn


# ── Endpoints ─────────────────────────────────────────────────────────
@app.get("/patents")
async def list_patents():
    return {"patents": _state["pat_list"]}


@app.get("/predict")
async def predict(patent: str, k: int = 10):
    p2i = _state["pat_to_idx"]
    if patent not in p2i:
        raise HTTPException(404, f"{patent} not found")
    q = p2i[patent]
    h = _state["h"]
    sims = h @ h[q]
    sims[q] = -np.inf
    top = np.argsort(-sims)[:k]
    return {
        "query": patent,
        "predictions": [
            {"rank": int(r+1), "patent_id": _state["pat_list"][int(i)],
             "cosine": float(sims[i])}
            for r, i in enumerate(top)
        ]
    }


@app.get("/explain")
async def explain(src: str, dst: str, top_k: int = 8):
    p2i = _state["pat_to_idx"]
    if src not in p2i: raise HTTPException(404, f"{src} not found")
    if dst not in p2i: raise HTTPException(404, f"{dst} not found")

    si, di = p2i[src], p2i[dst]
    h = _state["h"]
    cosine = float(h[si] @ h[di])

    with torch.no_grad():
        attn_layers = _extract_attn(_state["model"], _state["data"])

    last = attn_layers[-1]
    target_pats = {si, di}
    result_nodes, result_edges = [], []
    added_nodes = set()

    def add_node(ntype, nid):
        k = f"{ntype}_{nid}"
        if k in added_nodes: return
        added_nodes.add(k)
        pat_list = _state["pat_list"]
        co_list  = _state["co_list"]
        ipc_list = _state["ipc_list"]
        label = pat_list[nid] if ntype=="patent" else (co_list[nid][:20] if ntype=="company" else ipc_list[nid])
        role = "plaintiff" if (ntype=="patent" and nid==si) else ("defendant" if (ntype=="patent" and nid==di) else "context")
        result_nodes.append({"id": k, "type": ntype, "label": label, "role": role, "node_idx": nid})

    for nid in target_pats:
        add_node("patent", nid)

    for et_str, attn_data in last.items():
        src_t, rel, dst_t = et_str.split("|")
        ei    = torch.tensor(attn_data["edge_index"])
        alpha = torch.tensor(attn_data["alpha"])
        if src_t == "patent":
            mask = torch.isin(ei[0], torch.tensor(list(target_pats)))
        elif dst_t == "patent":
            mask = torch.isin(ei[1], torch.tensor(list(target_pats)))
        else: continue
        if mask.sum() == 0: continue
        rel_ei = ei[:, mask]; rel_a = alpha[mask]
        kk = min(top_k, mask.sum().item())
        for idx in rel_a.topk(kk).indices:
            s = int(rel_ei[0, idx]); d2 = int(rel_ei[1, idx]); w = float(rel_a[idx])
            add_node(src_t, s); add_node(dst_t, d2)
            result_edges.append({"from": f"{src_t}_{s}", "to": f"{dst_t}_{d2}",
                                  "rel": rel, "weight": w})

    return {"src": src, "dst": dst, "cosine": cosine,
            "nodes": result_nodes, "edges": result_edges}
'''

api_path = VIZ_DIR / 'app_explain.py'
api_path.write_text(FASTAPI_CODE.strip())
print(f'FastAPI 앱 저장 → {api_path}')
print('실행: uvicorn app_explain:app --reload --port 8000')

---
## Section 5 — Streamlit Dark Dashboard

```bash
streamlit run streamlit_viz.py
```

**UI 구성:**
- 좌측 사이드바: 특허 ID 입력 / 분석 모드 선택
- 우측 상단: 예측 결과 메트릭 카드 (Glassmorphism)
- 우측 하단: PyVis 인터랙티브 그래프 (Force-Directed, 네온)

In [ ]:
# Cell 14 — Streamlit 앱 파일 생성 (Galaxy + 대기실 아키텍처)
STREAMLIT_CODE = r'''
"""streamlit_viz.py — Patent Litigation Galaxy Dashboard"""
import json
from pathlib import Path

import streamlit as st
import streamlit.components.v1 as components
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pyvis.network import Network
from torch_geometric.nn import HeteroConv, GATConv
from streamlit_option_menu import option_menu
from streamlit_extras.metric_cards import style_metric_cards
from streamlit_extras.colored_header import colored_header

# ── Page Config ────────────────────────────────────────────
st.set_page_config(
    page_title="Patent Litigation Galaxy",
    page_icon="⚖️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Glassmorphism Dark CSS ─────────────────────────────────
st.markdown("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap');
  html, body, [class*="css"] {
    background-color: #050810 !important;
    color: #FFFFFF;
    font-family: 'JetBrains Mono', monospace;
  }
  [data-testid="stSidebar"] {
    background: linear-gradient(180deg, #070B18 0%, #0A0E1A 100%);
    border-right: 1px solid rgba(0,255,255,0.12);
  }
  .glass-card {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.09);
    border-radius: 16px;
    padding: 20px 24px;
    backdrop-filter: blur(14px);
    margin-bottom: 14px;
    transition: border-color 0.25s;
  }
  .glass-card:hover { border-color: rgba(0,255,255,0.3); }
  .metric-label { font-size:.72rem; color:#666; letter-spacing:.12em;
                  text-transform:uppercase; margin-bottom:4px; }
  .metric-cyan  { font-size:2.2rem; font-weight:700;
                  background:linear-gradient(135deg,#00FFFF,#00BFFF);
                  -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
  .metric-gold  { font-size:2.2rem; font-weight:700; color:#FFD700; }
  .metric-red   { font-size:2.2rem; font-weight:700; color:#FF4500; }
  /* Waiting list item */
  .wait-item {
    background: rgba(0,255,255,0.04);
    border: 1px solid rgba(0,255,255,0.12);
    border-radius: 10px;
    padding: 10px 14px;
    margin-bottom: 6px;
    cursor: pointer;
    transition: all 0.2s;
  }
  .wait-item:hover { border-color: rgba(0,255,255,0.5);
                     background: rgba(0,255,255,0.08); }
  .wait-item.selected {
    border-color: #00FFFF;
    background: rgba(0,255,255,0.14);
    box-shadow: 0 0 12px rgba(0,255,255,0.25);
  }
  .pid-tag { font-size:.82rem; font-weight:700; color:#00FFFF; }
  .role-tag-p { font-size:.68rem; color:#FFD700; border:1px solid #FFD70066;
                border-radius:4px; padding:1px 7px; }
  .role-tag-d { font-size:.68rem; color:#FF6B35; border:1px solid #FF6B3566;
                border-radius:4px; padding:1px 7px; }
  /* Button overrides */
  .stButton > button {
    background: linear-gradient(135deg,#00FFFF18,#00BFFF18);
    border: 1px solid #00FFFF55; color:#00FFFF;
    border-radius:8px; font-family:'JetBrains Mono',monospace;
    font-weight:700; transition:all 0.2s;
  }
  .stButton > button:hover {
    background:linear-gradient(135deg,#00FFFF33,#00BFFF33);
    border-color:#00FFFF; box-shadow:0 0 14px #00FFFF33;
  }
  .inject-btn > button {
    background: linear-gradient(135deg,#FF006622,#FF006644) !important;
    border-color:#FF0066 !important; color:#FF0066 !important;
  }
  .inject-btn > button:hover {
    box-shadow:0 0 16px #FF006666 !important;
  }
  hr { border-color:rgba(255,255,255,0.06) !important; }
  ::-webkit-scrollbar { width:5px; }
  ::-webkit-scrollbar-track { background:#050810; }
  ::-webkit-scrollbar-thumb { background:#00FFFF33; border-radius:3px; }
</style>
""", unsafe_allow_html=True)

# ── Constants ──────────────────────────────────────────────
HIDDEN_DIM, OUT_DIM, N_LAYERS, HEADS, FAISS_DIM = 256, 128, 3, 8, 1024
EDGE_TYPES = [
    ("patent","cites","patent"),("patent","cited_by","patent"),
    ("patent","has_ipc","ipc"),("ipc","rev_has_ipc","patent"),
    ("company","owns","patent"),("patent","owned_by","company"),
]
VIZ_DIR = Path(__file__).parent
CKPT    = VIZ_DIR / "gat_infonce_ckpt.pt"
DATA_PT = VIZ_DIR / "hetero_data.pt"

GALAXY_PHYSICS = json.dumps({
    "physics": {
        "solver": "forceAtlas2Based",
        "forceAtlas2Based": {
            "gravitationalConstant": -55, "centralGravity": 0.003,
            "springLength": 120, "springConstant": 0.04,
            "damping": 0.42, "avoidOverlap": 0.6,
        },
        "stabilization": {"iterations": 280, "updateInterval": 80, "fit": True},
        "maxVelocity": 60, "minVelocity": 0.8,
    },
    "interaction": {"hover": True, "tooltipDelay": 80,
                    "hideEdgesOnDrag": True, "navigationButtons": True},
    "nodes": {"shape": "dot"},
    "edges": {"smooth": {"type": "continuous"}},
})

# ── Model ──────────────────────────────────────────────────
class _Base(nn.Module):
    def __init__(self, pd, nc, ni, h, o, nl, dr):
        super().__init__()
        self.pat_proj = nn.Sequential(nn.Linear(pd,h), nn.LayerNorm(h), nn.ReLU())
        self.co_emb   = nn.Embedding(nc, h)
        self.ipc_emb  = nn.Embedding(ni, h)
        self.out_proj = nn.Linear(h, o)
        self.drop     = nn.Dropout(dr)
        self.norms    = nn.ModuleList([
            nn.ModuleDict({t:nn.LayerNorm(h) for t in ["patent","company","ipc"]})
            for _ in range(nl)
        ])
    def forward(self, d):
        x = {"patent":self.pat_proj(d["patent"].x),
             "company":self.co_emb.weight,"ipc":self.ipc_emb.weight}
        for c,n in zip(self.convs,self.norms):
            xn=c(x,d.edge_index_dict)
            x={k:self.drop(F.relu(n[k](xn[k]+x[k]))) if k in n
               else self.drop(F.relu(xn[k])) for k in xn}
        return {k:self.out_proj(v) for k,v in x.items()}

class HeteroGAT(_Base):
    def __init__(self,pd,nc,ni,h=HIDDEN_DIM,o=OUT_DIM,nl=N_LAYERS,heads=HEADS,dr=0.2):
        super().__init__(pd,nc,ni,h,o,nl,dr)
        hd=h//heads
        self.convs=nn.ModuleList([
            HeteroConv({et:GATConv(h,hd,heads=heads,dropout=dr,
                        concat=True,add_self_loops=False) for et in EDGE_TYPES},aggr="mean")
            for _ in range(nl)
        ])

# ── Load Everything ────────────────────────────────────────
@st.cache_resource(show_spinner="🌌 모델 & 갤럭시 로딩 중...")
def load_all():
    dev  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
    data = torch.load(DATA_PT, map_location="cpu", weights_only=False).to(dev)
    m    = HeteroGAT(FAISS_DIM, len(ckpt["co_list"]), len(ckpt["ipc_list"])).to(dev)
    m.load_state_dict(ckpt["model_state"]); m.eval()
    with torch.no_grad():
        h = F.normalize(m(data)["patent"], dim=1).cpu().numpy()

    # ── Train/Val vs Test 분리 ──
    pat_list  = ckpt["pat_list"]
    sued_e    = np.array(ckpt["sued_edges"])  # (2, N)
    sued_c    = ckpt["sued_cases"]
    test_cs   = set(ckpt["test_cases"])
    test_mask = np.array([c in test_cs for c in sued_c])

    test_pat_set   = set(sued_e[0,test_mask].tolist()) | set(sued_e[1,test_mask].tolist())
    galaxy_pat_set = set(range(len(pat_list))) - test_pat_set
    tv_plt_set     = set(sued_e[0,~test_mask].tolist())
    tv_def_set     = set(sued_e[1,~test_mask].tolist())

    # Test 특허 정보 (대기실 표시용)
    test_info = {}
    for i,(c,in_t) in enumerate(zip(sued_c,test_mask)):
        if not in_t: continue
        s,d = int(sued_e[0,i]), int(sued_e[1,i])
        for idx,role in [(s,"P"),(d,"D")]:
            entry = test_info.setdefault(idx,{"roles":set(),"cases":set()})
            entry["roles"].add(role); entry["cases"].add(c)

    return (m, data, h, ckpt, dev,
            galaxy_pat_set, test_pat_set, tv_plt_set, tv_def_set,
            test_info)


# ── Galaxy Graph Builder ───────────────────────────────────
def _build_galaxy(galaxy_pat_set, tv_plt_set, tv_def_set,
                  ckpt, height="820px",
                  test_pat_idx=None, predicted_edges=None):
    pat_list   = ckpt["pat_list"]
    co_list    = ckpt["co_list"]
    ipc_list   = ckpt["ipc_list"]
    pat_to_idx = ckpt["pat_to_idx"]
    ipc_to_idx = ckpt["ipc_to_idx"]
    co_to_idx  = ckpt["co_to_idx"]
    df_cit     = ckpt["df_cit"]
    df_ipc     = ckpt["df_ipc"]
    df_asgn    = ckpt["df_asgn"]
    litigated  = tv_plt_set | tv_def_set

    net = Network(height=height, width="100%", bgcolor="#050810",
                  font_color="#FFFFFF", directed=True,
                  notebook=False, cdn_resources="in_line")

    for idx in galaxy_pat_set:
        pid   = str(pat_list[idx])
        is_p  = idx in tv_plt_set
        is_d  = idx in tv_def_set
        if is_p and is_d: color,sz = "#FFA500",26
        elif is_p:         color,sz = "#FFD700",22
        elif is_d:         color,sz = "#FF6B35",22
        else:              color,sz = "#007B9E",12
        role = "P&D" if (is_p and is_d) else ("Plaintiff" if is_p else ("Defendant" if is_d else "•"))
        net.add_node(f"p_{idx}", label=pid,
            title=f"<b>[PATENT]</b><br>{pid}<br><span style='color:{color}'>◉ {role}</span>",
            color={"background":color,"border":color,
                   "highlight":{"background":"#FFFFFF","border":"#FFFFFF"},
                   "hover":{"background":"#FFFFFF","border":"#FFFFFF"}},
            size=sz,
            font={"color":"#555555" if role=="•" else "#FFFFFF","size":9 if role=="•" else 11},
            borderWidth=1,
            shadow={"enabled":True,"color":color+"44","size":12 if role!="•" else 4,"x":0,"y":0})

    added_e = set()
    for _, row in df_cit.iterrows():
        si = pat_to_idx.get(str(row.get("citing_patent", row.get("patent_id",""))))
        di = pat_to_idx.get(str(row.get("cited_patent",  row.get("cited_id",""))))
        if si is None or di is None: continue
        if si not in galaxy_pat_set or di not in galaxy_pat_set: continue
        if (si,di) in added_e: continue
        added_e.add((si,di))
        net.add_edge(f"p_{si}",f"p_{di}",
            color={"color":"#002A44","hover":"#00BFFF"},
            width=0.9, arrows={"to":{"enabled":True,"scaleFactor":0.35}},
            smooth={"type":"continuous"})

    ipc_g={}
    for _,row in df_ipc.iterrows():
        pi=pat_to_idx.get(str(row.get("patent_id","")))
        ci=ipc_to_idx.get(str(row.get("ipc_code","")))
        if pi is None or ci is None: continue
        if pi not in litigated or pi not in galaxy_pat_set: continue
        ipc_g[ci]=str(row.get("ipc_code",""))
        net.add_edge(f"p_{pi}",f"i_{ci}",
            color={"color":"#0F2200","hover":"#7FFF00"},
            width=0.7, arrows={"to":{"enabled":False}}, smooth={"type":"continuous"})
    for ci,ic in ipc_g.items():
        net.add_node(f"i_{ci}",label=ic,
            title=f"<b>[IPC]</b><br>{ic}",
            color={"background":"#132200","border":"#32CD32","hover":{"background":"#32CD32"}},
            size=10,shape="diamond",font={"color":"#32CD32","size":8})

    co_g={}
    for _,row in df_asgn.iterrows():
        pi=pat_to_idx.get(str(row.get("patent_id","")))
        co_s=str(row.get("assignee",row.get("company","")))
        ci2=co_to_idx.get(co_s)
        if pi is None or ci2 is None: continue
        if pi not in litigated or pi not in galaxy_pat_set: continue
        co_g[ci2]=co_s
        net.add_edge(f"co_{ci2}",f"p_{pi}",
            color={"color":"#1A0025","hover":"#FF69B4"},
            width=0.7, arrows={"to":{"enabled":True,"scaleFactor":0.3}},
            smooth={"type":"continuous"})
    for ci2,co_s in co_g.items():
        lb=co_s[:16]+".." if len(co_s)>16 else co_s
        net.add_node(f"co_{ci2}",label=lb,
            title=f"<b>[COMPANY]</b><br>{co_s}",
            color={"background":"#1A0030","border":"#FF00FF","hover":{"background":"#FF00FF"}},
            size=13,shape="square",font={"color":"#FF00FF","size":8})

    if test_pat_idx is not None:
        tpid = str(pat_list[test_pat_idx])
        net.add_node(f"p_{test_pat_idx}", label=tpid,
            title=f"<b>🛸 [TEST — INCOMING]</b><br>{tpid}<br>갤럭시에 방금 주입된 신규 노드",
            color={"background":"#FFFFFF","border":"#00FFFF",
                   "highlight":{"background":"#00FFFF","border":"#FFFFFF"}},
            size=52, font={"color":"#FFFFFF","size":14,"bold":True},
            borderWidth=4,
            shadow={"enabled":True,"color":"#FFFFFF99","size":40,"x":0,"y":0})
        if predicted_edges:
            mx = max(c for _,c in predicted_edges)
            for rank,(di,cos) in enumerate(predicted_edges,1):
                if f"p_{di}" not in {n["id"] for n in net.nodes}: continue
                w = 2 + (cos/(mx+1e-8))*13
                dc = "#FF0066" if cos>0.75 else "#FF6600" if cos>0.55 else "#FFAA00"
                net.add_edge(f"p_{test_pat_idx}",f"p_{di}",
                    label=f"#{rank} {cos:.3f}",
                    font={"color":dc,"size":10,"bold":True,"align":"middle"},
                    title=f"<b>⚠️ 소송 예측</b><br>Cosine: {cos:.4f}  Rank: {rank}",
                    width=w,
                    color={"color":dc,"highlight":"#FFFFFF","hover":"#FFFFFF"},
                    arrows={"to":{"enabled":True,"scaleFactor":0.9}},
                    smooth={"type":"curvedCW","roundness":0.3},
                    shadow={"enabled":True,"color":dc+"CC","size":18,"x":0,"y":0})

    net.set_options(GALAXY_PHYSICS)
    return net


# ── Session State Init ─────────────────────────────────────
if "selected_test_idx" not in st.session_state:
    st.session_state.selected_test_idx = None
if "search_query" not in st.session_state:
    st.session_state.search_query = ""

# ── Load Data ──────────────────────────────────────────────
(model, data, h_emb, ckpt, dev,
 galaxy_pat_set, test_pat_set, tv_plt_set, tv_def_set,
 test_info) = load_all()

pat_list   = ckpt["pat_list"]
co_list    = ckpt["co_list"]
ipc_list   = ckpt["ipc_list"]
pat_to_idx = ckpt["pat_to_idx"]

# ─────────────────────────────────────────────────────────
#  SIDEBAR
# ─────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(
        "<h2 style='color:#00FFFF;text-shadow:0 0 18px #00FFFF99;"
        "margin-bottom:2px;font-size:1.35rem'>⚖️ PatentNet Galaxy</h2>"
        "<p style='color:#444;font-size:0.72rem;margin:0 0 10px'>GAT + InfoNCE  |  Litigation Predictor</p>",
        unsafe_allow_html=True)
    st.divider()

    mode = option_menu(
        menu_title=None,
        options=["🌌 Galaxy", "📊 Statistics"],
        icons=["stars","bar-chart-fill"],
        default_index=0,
        styles={
            "container":         {"background-color":"transparent","padding":"0"},
            "icon":              {"color":"#00FFFF","font-size":"13px"},
            "nav-link":          {"color":"#888","font-size":"13px",
                                  "border-radius":"8px","margin":"2px 0"},
            "nav-link-selected": {"background-color":"rgba(0,255,255,0.1)",
                                  "color":"#00FFFF","font-weight":"700",
                                  "border":"1px solid rgba(0,255,255,0.25)"},
        })

    st.divider()

    if mode == "🌌 Galaxy":
        # ── 대기실 헤더 ──
        st.markdown(
            f"<div style='color:#888;font-size:.72rem;letter-spacing:.1em;"
            f"text-transform:uppercase;margin-bottom:8px'>"
            f"🧪 TEST 특허 대기실 &nbsp;"
            f"<span style='color:#00FFFF'>{len(test_pat_set)}개</span></div>",
            unsafe_allow_html=True)

        search = st.text_input("", placeholder="특허 번호 검색...", key="search_query",
                               label_visibility="collapsed")

        # 대기실 목록 필터
        test_sorted = sorted(test_pat_set,
                             key=lambda i: ("P" in test_info.get(i,{}).get("roles",set())), reverse=True)
        if search:
            test_sorted = [i for i in test_sorted
                           if search.lower() in str(pat_list[i]).lower()]

        # 현재 선택 초기화 버튼
        sel = st.session_state.selected_test_idx
        if sel is not None:
            c_res, c_sel = st.columns([1,2])
            with c_res:
                if st.button("✕ 초기화", use_container_width=True):
                    st.session_state.selected_test_idx = None
                    st.rerun()
            with c_sel:
                st.markdown(
                    f"<div style='color:#00FFFF;font-size:.78rem;padding:6px 0'>"
                    f"선택됨: <b>{str(pat_list[sel])[:18]}</b></div>",
                    unsafe_allow_html=True)
            st.divider()

        # ── 특허 카드 목록 ──
        MAX_SHOW = 40
        shown = test_sorted[:MAX_SHOW]
        for idx in shown:
            pid     = str(pat_list[idx])
            roles   = test_info.get(idx,{}).get("roles", set())
            n_cases = len(test_info.get(idx,{}).get("cases", set()))
            is_sel  = (idx == st.session_state.selected_test_idx)

            role_tags = ""
            if "P" in roles:
                role_tags += "<span class='role-tag-p'>P</span> "
            if "D" in roles:
                role_tags += "<span class='role-tag-d'>D</span>"

            border = "#00FFFF" if is_sel else "rgba(0,255,255,0.12)"
            bg     = "rgba(0,255,255,0.10)" if is_sel else "rgba(0,255,255,0.03)"

            st.markdown(
                f"<div class='wait-item' style='border-color:{border};background:{bg}'>"
                f"<div class='pid-tag'>{pid}</div>"
                f"<div style='font-size:.68rem;color:#555;margin-top:3px'>"
                f"{role_tags} &nbsp; {n_cases}건</div></div>",
                unsafe_allow_html=True)

            btn_label = "🛸 주입" if not is_sel else "✓ 선택됨"
            btn_key   = f"inject_{idx}"
            if is_sel:
                if st.button(btn_label, key=btn_key, use_container_width=True, disabled=True):
                    pass
            else:
                if st.button(btn_label, key=btn_key, use_container_width=True):
                    st.session_state.selected_test_idx = idx
                    st.rerun()

        if len(test_sorted) > MAX_SHOW:
            st.caption(f"... {len(test_sorted)-MAX_SHOW}개 더 있음 (검색으로 필터)")

        st.divider()
        st.markdown("""
<div style='font-size:.72rem;color:#444;line-height:1.9'>
<b style='color:#666'>노드</b><br>
⬤ <span style='color:#FFD700'>Gold</span> 원고특허<br>
⬤ <span style='color:#FF6B35'>Orange</span> 피고특허<br>
⬤ <span style='color:#007B9E'>Blue</span> 연결특허<br>
◆ <span style='color:#32CD32'>Green</span> IPC<br>
■ <span style='color:#FF00FF'>Magenta</span> 회사<br>
⬤ <span style='color:#FFFFFF'>White★</span> 주입된 Test<br><br>
<b style='color:#666'>예측 엣지</b><br>
━ <span style='color:#FF0066'>Red</span> HIGH (>0.75)<br>
━ <span style='color:#FF6600'>Orange</span> MED (>0.55)<br>
━ <span style='color:#FFAA00'>Yellow</span> LOW
</div>""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────
#  MAIN AREA
# ─────────────────────────────────────────────────────────
sel_idx = st.session_state.selected_test_idx

if mode == "🌌 Galaxy":

    # ── Case A: 초기 갤럭시 (대기 상태) ─────────
    if sel_idx is None:
        colored_header(
            label="소송 특허 생태계 — Galaxy View",
            description=f"Train+Val {len(galaxy_pat_set)}개 특허 | 좌측에서 Test 특허를 선택해 주입하세요",
            color_name="blue-70")

        # 갤럭시 렌더링
        with st.spinner("🌌 갤럭시 렌더링 중..."):
            gnet = _build_galaxy(
                galaxy_pat_set, tv_plt_set, tv_def_set, ckpt,
                height="820px")
            components.html(gnet.generate_html(), height=840, scrolling=False)

    # ── Case B: Test 노드 주입! ───────────────────
    else:
        test_pid = str(pat_list[sel_idx])

        # 예측 계산
        sims = h_emb @ h_emb[sel_idx]
        sims[sel_idx] = -np.inf
        for ti in test_pat_set:   # 갤럭시 내 노드만 후보
            sims[ti] = -np.inf
        K = 7
        top_idx    = np.argsort(-sims)[:K]
        pred_edges = [(int(i), float(sims[i])) for i in top_idx]
        best_cos   = float(sims[top_idx[0]])

        verdict, vc = (
            ("HIGH RISK", "#FF0066") if best_cos > 0.75 else
            ("MEDIUM",    "#FF6600") if best_cos > 0.55 else
            ("LOW",       "#FFAA00")
        )
        roles = test_info.get(sel_idx,{}).get("roles",set())
        role_str = "/".join(sorted(roles)) if roles else "—"

        # ── 메트릭 카드 ──
        colored_header(
            label=f"🛸 {test_pid} — 갤럭시 주입 완료",
            description=f"Test 특허가 기존 소송 네트워크에 주입되었습니다  |  Top-1 cosine: {best_cos:.4f}",
            color_name="blue-70")

        c1, c2, c3, c4 = st.columns(4)
        c1.markdown(
            f'<div class="glass-card"><div class="metric-label">Test 특허</div>'
            f'<div class="metric-cyan" style="font-size:1.1rem">{test_pid}</div></div>',
            unsafe_allow_html=True)
        c2.markdown(
            f'<div class="glass-card"><div class="metric-label">소송 역할</div>'
            f'<div class="metric-gold">{role_str}</div></div>',
            unsafe_allow_html=True)
        c3.markdown(
            f'<div class="glass-card"><div class="metric-label">Top-1 Cosine</div>'
            f'<div class="metric-cyan">{best_cos:.4f}</div></div>',
            unsafe_allow_html=True)
        c4.markdown(
            f'<div class="glass-card"><div class="metric-label">Risk Level</div>'
            f'<div style="font-size:1.8rem;font-weight:700;color:{vc};'
            f'text-shadow:0 0 14px {vc}88">{verdict}</div></div>',
            unsafe_allow_html=True)
        style_metric_cards(background_color="#050810", border_color="#00FFFF18")

        # ── 갤럭시 + 주입 그래프 & 예측 테이블 ──
        col_g, col_t = st.columns([3, 1])
        with col_g:
            colored_header(
                label="갤럭시 주입 시뮬레이션",
                description="⚡ 네온 핑크 엣지 = 소송 예측 링크",
                color_name="blue-70")
            with st.spinner("⚡ 주입 렌더링 중..."):
                gnet = _build_galaxy(
                    galaxy_pat_set, tv_plt_set, tv_def_set, ckpt,
                    height="820px",
                    test_pat_idx=sel_idx,
                    predicted_edges=pred_edges)
                components.html(gnet.generate_html(), height=840, scrolling=False)

        with col_t:
            colored_header(label="예측 피고 Top-7",
                           description="갤럭시 내 cosine 상위",
                           color_name="violet-70")
            rows = []
            for rank,(di,cos) in enumerate(pred_edges,1):
                risk = "HIGH" if cos>0.75 else "MED" if cos>0.55 else "LOW"
                rows.append({"#":rank, "Patent":str(pat_list[di]),
                             "Cosine":f"{cos:.4f}","Risk":risk})
            df_pred = pd.DataFrame(rows)
            st.dataframe(df_pred, use_container_width=True,
                         hide_index=True, height=370)

            st.divider()
            colored_header(label="테스트 케이스", description="", color_name="violet-70")
            cases = test_info.get(sel_idx,{}).get("cases",set())
            for c in sorted(cases):
                st.markdown(
                    f"<div class='glass-card' style='padding:10px 14px'>"
                    f"<div style='color:#888;font-size:.7rem'>CASE</div>"
                    f"<div style='color:#00FFFF;font-size:.9rem;font-weight:700'>{c}</div>"
                    f"</div>", unsafe_allow_html=True)

# ── Statistics Mode ────────────────────────────────────────
else:
    colored_header(label="데이터셋 통계",
                   description="GAT + InfoNCE 학습 기준",
                   color_name="blue-70")

    c1,c2,c3,c4 = st.columns(4)
    stats = [
        ("전체 특허",         len(pat_list),           "#00FFFF"),
        ("Galaxy (Train+Val)",len(galaxy_pat_set),     "#FFD700"),
        ("Waiting Room (Test)",len(test_pat_set),      "#FF0066"),
        ("소송 케이스",        269,                     "#FF6B35"),
    ]
    for col,(lb,v,color) in zip([c1,c2,c3,c4],stats):
        col.markdown(
            f'<div class="glass-card"><div class="metric-label">{lb}</div>'
            f'<div style="font-size:2.2rem;font-weight:700;color:{color};'
            f'text-shadow:0 0 10px {color}55">{v:,}</div></div>',
            unsafe_allow_html=True)

    colored_header(label="모델 성능 비교",
                   description="Test set (27 cases, 165 pairs) 기준",
                   color_name="violet-70")
    perf = pd.DataFrame({
        "Model":       ["FAISS (baseline)","SAGE+Margin","SAGE+InfoNCE","GAT+Margin","GAT+InfoNCE ★"],
        "AUROC":       [0.9301,0.9070,0.8563,0.8950,0.9193],
        "AUPRC":       [0.8961,0.8585,0.7876,0.8571,0.8851],
        "MRR":         [0.1817,0.1213,0.0925,0.0993,0.1271],
        "Hits@10":     [0.5152,0.3394,0.2000,0.2364,0.3394],
        "Median Rank": [10,41,89,36,28],
    })
    st.dataframe(perf, use_container_width=True, hide_index=True)
    st.caption("★ 현재 Galaxy 대시보드에서 사용 중인 모델")

    colored_header(label="Train/Val/Test 분할",
                   description="케이스 수준 분할 (데이터 누수 방지)",
                   color_name="blue-70")
    split_df = pd.DataFrame({
        "Split":    ["Train","Val","Test"],
        "Cases":    [215,27,27],
        "Sued Pairs":[2198,145,165],
        "역할":     ["Galaxy의 핵심","Galaxy에 포함","대기실 후보"],
    })
    st.dataframe(split_df, use_container_width=True, hide_index=True)
'''

st_path = VIZ_DIR / 'streamlit_viz.py'
st_path.write_text(STREAMLIT_CODE.strip())
print(f'Streamlit Galaxy 앱 저장 → {st_path}')
print()
print('실행: streamlit run streamlit_viz.py')

In [ ]:
# Cell 15 — Launch Instructions
print('=' * 60)
print('  Patent Litigation Visualization System — 실행 가이드')
print('=' * 60)
print()
print('[STEP 1] gnn_266_v2.ipynb 체크포인트 저장 (최초 1회)')
print('  → Cell S 내용을 복사하여 Cell 12 다음에 추가 후 실행')
print()
print('[STEP 2] 이 노트북 전체 실행')
print(f'  → 체크포인트: {CKPT}')
print(f'  → HeteroData: {DATA_PT}')
print()
print('[STEP 3] Streamlit 실행 (권장 — 올인원 대시보드)')
print(f'  cd {VIZ_DIR}')
print('  streamlit run streamlit_viz.py')
print()
print('[STEP 4 - 선택] FastAPI 서버 별도 실행')
print(f'  cd {VIZ_DIR}')
print('  uvicorn app_explain:app --reload --port 8000')
print('  → http://localhost:8000/docs 에서 API 테스트 가능')
print()
print('생성된 파일:')
for f in [OUT_HTML, VIZ_DIR/'attention_heatmap.png',
           VIZ_DIR/'app_explain.py', VIZ_DIR/'streamlit_viz.py']:
    mark = '✓' if Path(f).exists() else '○'
    print(f'  {mark} {f}')